# 01 — Zone Definition

Defines census tracts as the unit of analysis and derives the **Y variable** (`zone_type`) from PLUTO `landuse`.

**Method:**
1. Load PLUTO, filter to selected boroughs (default: Manhattan)
2. Group lots by `bct2020` (2020 Census Tract)
3. Compute area-weighted `landuse` distribution per tract
4. Map PLUTO landuse codes to 6 zone-type categories via plurality rule
5. Compute tract centroids from lot coordinates

**Output columns:** `tract_id`, `borough`, `tract_lat`, `tract_lon`, `zone_type`, `tract_lot_count`

**Output file:** `csv/01_zone_definition.csv`

In [ ]:
# ── Papermill parameters ──────────────────────────────
ZONES_CONFIG = "zones.json"

In [ ]:
import pandas as pd
import numpy as np
import json
import os

os.makedirs("csv", exist_ok=True)

with open(ZONES_CONFIG, encoding="utf-8") as f:
    config = json.load(f)

BOROUGH_FILTER = config["borough_filter"]
BOROUGH_CODES = config["borough_codes"]
PLUTO_PATH = config["pluto_path"]

# Map borough abbreviations to numeric codes for filtering
boro_code_filter = [str(BOROUGH_CODES[b]) for b in BOROUGH_FILTER]
print(f"Filtering boroughs: {BOROUGH_FILTER} (codes: {boro_code_filter})")
print(f"PLUTO path: {PLUTO_PATH}")

In [ ]:
# ── Load PLUTO (only needed columns) ──────────────────
PLUTO_COLS = [
    "borocode", "bct2020", "landuse", "lotarea",
    "latitude", "longitude", "borough",
]

df_pluto = pd.read_csv(PLUTO_PATH, usecols=PLUTO_COLS, dtype={"landuse": str, "bct2020": str})
df_pluto = df_pluto[df_pluto["borocode"].astype(str).isin(boro_code_filter)].copy()

print(f"PLUTO rows after borough filter: {len(df_pluto):,}")
print(f"Unique tracts: {df_pluto['bct2020'].nunique()}")
print(f"\nLanduse value counts:")
print(df_pluto["landuse"].value_counts().to_string())

In [ ]:
# ── Map PLUTO landuse codes to zone type categories ───
#
# PLUTO landuse codes:
#   01 = One & Two Family Buildings
#   02 = Multi-Family Walk-Up Buildings
#   03 = Multi-Family Elevator Buildings
#   04 = Mixed Residential & Commercial
#   05 = Commercial & Office Buildings
#   06 = Industrial & Manufacturing
#   07 = Transportation & Utility
#   08 = Public Facilities & Institutions
#   09 = Open Space & Outdoor Recreation
#   10 = Parking Facilities
#   11 = Vacant Land

LANDUSE_TO_ZONE = {
    "01": "Residential",
    "02": "Residential",
    "03": "Residential",     # multi-family elevator = still residential
    "04": "Mixed-Use",       # mixed res + commercial
    "05": "Commercial",
    "06": "Industrial",
    "07": "Infrastructure",  # transportation & utility
    "08": "Institutional",   # public facilities
    "09": "Open Space",
    "10": "Infrastructure",  # parking
    "11": "Infrastructure",  # vacant
}

# Minimum plurality ratio to assign a dominant type;
# below this threshold the tract is classified as Mixed-Use
PLURALITY_THRESHOLD = 0.40

print("Landuse → Zone Type mapping:")
for lu, zt in sorted(LANDUSE_TO_ZONE.items()):
    print(f"  {lu} → {zt}")

In [ ]:
# ── Compute area-weighted zone type per tract ─────────

df_pluto["lotarea"] = pd.to_numeric(df_pluto["lotarea"], errors="coerce").fillna(0)
df_pluto["latitude"] = pd.to_numeric(df_pluto["latitude"], errors="coerce")
df_pluto["longitude"] = pd.to_numeric(df_pluto["longitude"], errors="coerce")

# Map each lot's landuse to a zone category
df_pluto["zone_cat"] = df_pluto["landuse"].map(LANDUSE_TO_ZONE).fillna("Unknown")

# Aggregate per tract
tract_records = []

for tract_id, group in df_pluto.groupby("bct2020"):
    total_area = group["lotarea"].sum()
    lot_count = len(group)
    
    # Area-weighted centroid
    valid_coords = group.dropna(subset=["latitude", "longitude"])
    if len(valid_coords) == 0:
        continue
    tract_lat = valid_coords["latitude"].mean()
    tract_lon = valid_coords["longitude"].mean()
    boro = group["borough"].mode().iloc[0] if len(group["borough"].mode()) > 0 else "Unknown"
    
    # Zone type by area-weighted plurality
    if total_area == 0:
        zone_type = "Mixed-Use"
    else:
        zone_area = group.groupby("zone_cat")["lotarea"].sum()
        dominant = zone_area.idxmax()
        dominant_ratio = zone_area[dominant] / total_area
        
        if dominant == "Infrastructure" or dominant == "Unknown":
            # Rare categories — fall back to second-most dominant or Mixed-Use
            remaining = zone_area.drop(["Infrastructure", "Unknown"], errors="ignore")
            if len(remaining) > 0:
                dominant = remaining.idxmax()
                dominant_ratio = remaining[dominant] / total_area
            else:
                zone_type = "Mixed-Use"
                dominant_ratio = 0
        
        if dominant_ratio >= PLURALITY_THRESHOLD:
            zone_type = dominant
        else:
            zone_type = "Mixed-Use"
    
    tract_records.append({
        "tract_id": tract_id,
        "borough": boro,
        "tract_lat": round(tract_lat, 7),
        "tract_lon": round(tract_lon, 7),
        "zone_type": zone_type,
        "tract_lot_count": lot_count,
    })

df_zones = pd.DataFrame(tract_records)
print(f"Tracts defined: {len(df_zones)}")
print(f"\nZone type distribution:")
print(df_zones["zone_type"].value_counts().to_string())

In [ ]:
# ── Save output ───────────────────────────────────────
output_path = "csv/01_zone_definition.csv"
df_zones.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved: {output_path}  ({len(df_zones)} rows x {df_zones.shape[1]} cols)")
df_zones.head(10)

In [ ]:
# ── Summary statistics ────────────────────────────────
print("Zone type breakdown:")
for zt in df_zones["zone_type"].unique():
    subset = df_zones[df_zones["zone_type"] == zt]
    print(f"  {zt:<20s} {len(subset):>4d} tracts  "
          f"(avg {subset['tract_lot_count'].mean():.0f} lots/tract)")

print(f"\nLat range: {df_zones['tract_lat'].min():.4f} – {df_zones['tract_lat'].max():.4f}")
print(f"Lon range: {df_zones['tract_lon'].min():.4f} – {df_zones['tract_lon'].max():.4f}")